In [4]:
import requests
import pandas as pd
import time

API_KEY = "6d0c90e0ba689394edec2004c587f13996dbc4c2b5773ffe25c5236cc93262fa"

headers = {
    "X-API-Key": API_KEY
}

url_locations = "https://api.openaq.org/v3/locations"

params = {
    "bbox": "-74.3,40.5,-73.6,41",
    "limit": 1000
}

response = requests.get(url_locations, headers=headers, params=params, timeout=20)
data = response.json()

locations = data.get("results", [])

sensor_ids = []

for loc in locations:
    for s in loc.get("sensors", []):
        sensor_ids.append(s["id"])

sensor_ids = list(set(sensor_ids))

print("Всего сенсоров:", len(sensor_ids))


def get_sensor_data(sensor_id):
    url = f"https://api.openaq.org/v3/sensors/{sensor_id}/days"

    params = {
        "date_from": "2024-01-01",
        "date_to": "2024-12-31",
        "limit": 1000
    }

    try:
        r = requests.get(url, headers=headers, params=params, timeout=15)

        if r.status_code != 200:
            print(f"Sensor {sensor_id} failed: {r.status_code}")
            return None

        data = r.json()

        results = data.get("results", [])
        if not results:
            return None

        df = pd.json_normalize(results)
        df["sensor_id"] = sensor_id

        return df

    except requests.exceptions.Timeout:
        print(f"Timeout sensor {sensor_id}")
        return None

    except Exception as e:
        print(f"Error sensor {sensor_id}: {e}")
        return None


all_data = []

max_sensors = len(sensor_ids)

for i, sid in enumerate(sensor_ids):

    print(f"Processing {i+1}/{max_sensors} → sensor {sid}")

    df = get_sensor_data(sid)

    if df is not None:
        all_data.append(df)

    time.sleep(0.1)


if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
else:
    final_df = pd.DataFrame()

print("DONE")
print("Rows:", len(final_df))
print(final_df.head())

Всего сенсоров: 249
Processing 1/249 → sensor 1536
Processing 2/249 → sensor 11571202
Processing 3/249 → sensor 11074566
Processing 4/249 → sensor 11074571
Processing 5/249 → sensor 7979020
Processing 6/249 → sensor 11074572
Processing 7/249 → sensor 15101967
Processing 8/249 → sensor 15101968
Processing 9/249 → sensor 15101969
Processing 10/249 → sensor 15101970
Processing 11/249 → sensor 15101971
Processing 12/249 → sensor 10454036
Processing 13/249 → sensor 11074582
Processing 14/249 → sensor 13322774
Processing 15/249 → sensor 13322775
Processing 16/249 → sensor 11074585
Processing 17/249 → sensor 13322776
Processing 18/249 → sensor 10454043
Processing 19/249 → sensor 11074587
Processing 20/249 → sensor 13322777
Processing 21/249 → sensor 7971870
Processing 22/249 → sensor 13322778
Processing 23/249 → sensor 7254050
Processing 24/249 → sensor 13719588
Processing 25/249 → sensor 13719589
Processing 26/249 → sensor 13719590
Processing 27/249 → sensor 13719591
Processing 28/249 → sens

In [5]:
final_df["date"] = pd.to_datetime(final_df["coverage.datetimeTo.utc"])

final_df["parameter"] = final_df["parameter.name"]

aq = final_df[[
    "sensor_id",
    "date",
    "parameter",
    "value"
]].copy()

aq


,sensor_id,date,parameter,value
0,1536,2024-01-02 05:00:00+00:00,co,0.2620
1,1536,2024-01-03 05:00:00+00:00,co,0.3270
2,1536,2024-01-04 05:00:00+00:00,co,0.2910
3,1536,2024-01-05 05:00:00+00:00,co,0.2240
4,1536,2024-01-06 05:00:00+00:00,co,0.2770
...,...,...,...,...
30246,1535,2024-12-28 05:00:00+00:00,no2,0.0397
30247,1535,2024-12-29 04:00:00+00:00,no2,0.0425
30248,1535,2024-12-30 05:00:00+00:00,no2,0.0233
30249,1535,2024-12-31 05:00:00+00:00,no2,0.0112


In [6]:
response = requests.get(url_locations, headers=headers, params=params)
data = response.json()

locations = data["results"]

In [7]:
locations_list = []

for loc in locations:
    if loc.get("coordinates") and "sensors" in loc:
        for s in loc["sensors"]:
            locations_list.append({
                "sensor_id": s["id"],
                "lat": loc["coordinates"]["latitude"],
                "lon": loc["coordinates"]["longitude"],
                "location": loc["name"]})

locations_df = pd.DataFrame(locations_list)
locations_df

,sensor_id,lat,lon,location
0,671,40.819700,-73.948100,CCNY
1,673,40.819700,-73.948100,CCNY
2,674,40.596900,-74.126400,Susan Wagner
3,1097,40.849200,-73.931900,Manhattan/IS143
4,1098,40.816101,-73.902199,Bronx - IS52
...,...,...,...,...
244,16083421,40.810157,-73.960457,"Teachers College, Columbia University"
245,16083422,40.810157,-73.960457,"Teachers College, Columbia University"
246,16083423,40.810157,-73.960457,"Teachers College, Columbia University"
247,16083424,40.810157,-73.960457,"Teachers College, Columbia University"


In [8]:
final_df["date"] = pd.to_datetime(final_df["coverage.datetimeTo.utc"])

final_df["parameter"] = final_df["parameter.name"]

aq_df = final_df.merge(locations_df, on="sensor_id", how="left")
aq_df

,value,coordinates,flagInfo.hasFlags,parameter.id,parameter.name,parameter.units,parameter.displayName,period.label,period.interval,period.datetimeFrom.utc,...,coverage.datetimeFrom.utc,coverage.datetimeFrom.local,coverage.datetimeTo.utc,coverage.datetimeTo.local,sensor_id,date,parameter,lat,lon,location
0,0.2620,None,False,8,co,ppm,None,1day,24:00:00,2024-01-01T05:00:00Z,...,2024-01-01T06:00:00Z,2024-01-01T01:00:00-05:00,2024-01-02T05:00:00Z,2024-01-02T00:00:00-05:00,1536,2024-01-02 05:00:00+00:00,co,40.85355,-73.9661,Fort Lee Near Road
1,0.3270,None,False,8,co,ppm,None,1day,24:00:00,2024-01-02T05:00:00Z,...,2024-01-02T06:00:00Z,2024-01-02T01:00:00-05:00,2024-01-03T05:00:00Z,2024-01-03T00:00:00-05:00,1536,2024-01-03 05:00:00+00:00,co,40.85355,-73.9661,Fort Lee Near Road
2,0.2910,None,False,8,co,ppm,None,1day,24:00:00,2024-01-03T05:00:00Z,...,2024-01-03T06:00:00Z,2024-01-03T01:00:00-05:00,2024-01-04T05:00:00Z,2024-01-04T00:00:00-05:00,1536,2024-01-04 05:00:00+00:00,co,40.85355,-73.9661,Fort Lee Near Road
3,0.2240,None,False,8,co,ppm,None,1day,24:00:00,2024-01-04T05:00:00Z,...,2024-01-04T06:00:00Z,2024-01-04T01:00:00-05:00,2024-01-05T05:00:00Z,2024-01-05T00:00:00-05:00,1536,2024-01-05 05:00:00+00:00,co,40.85355,-73.9661,Fort Lee Near Road
4,0.2770,None,False,8,co,ppm,None,1day,24:00:00,2024-01-05T05:00:00Z,...,2024-01-05T06:00:00Z,2024-01-05T01:00:00-05:00,2024-01-06T05:00:00Z,2024-01-06T00:00:00-05:00,1536,2024-01-06 05:00:00+00:00,co,40.85355,-73.9661,Fort Lee Near Road
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30246,0.0397,None,False,7,no2,ppm,None,1day,24:00:00,2024-12-27T05:00:00Z,...,2024-12-27T06:00:00Z,2024-12-27T01:00:00-05:00,2024-12-28T05:00:00Z,2024-12-28T00:00:00-05:00,1535,2024-12-28 05:00:00+00:00,no2,40.85355,-73.9661,Fort Lee Near Road
30247,0.0425,None,False,7,no2,ppm,None,1day,24:00:00,2024-12-28T05:00:00Z,...,2024-12-28T06:00:00Z,2024-12-28T01:00:00-05:00,2024-12-29T04:00:00Z,2024-12-28T23:00:00-05:00,1535,2024-12-29 04:00:00+00:00,no2,40.85355,-73.9661,Fort Lee Near Road
30248,0.0233,None,False,7,no2,ppm,None,1day,24:00:00,2024-12-29T05:00:00Z,...,2024-12-29T09:00:00Z,2024-12-29T04:00:00-05:00,2024-12-30T05:00:00Z,2024-12-30T00:00:00-05:00,1535,2024-12-30 05:00:00+00:00,no2,40.85355,-73.9661,Fort Lee Near Road
30249,0.0112,None,False,7,no2,ppm,None,1day,24:00:00,2024-12-30T05:00:00Z,...,2024-12-30T06:00:00Z,2024-12-30T01:00:00-05:00,2024-12-31T05:00:00Z,2024-12-31T00:00:00-05:00,1535,2024-12-31 05:00:00+00:00,no2,40.85355,-73.9661,Fort Lee Near Road


In [9]:
aq = aq_df[[
    "sensor_id",
    "date",
    "parameter",
    "value",
    "lat",
    "lon",
    "location"]]
aq

,sensor_id,date,parameter,value,lat,lon,location
0,1536,2024-01-02 05:00:00+00:00,co,0.2620,40.85355,-73.9661,Fort Lee Near Road
1,1536,2024-01-03 05:00:00+00:00,co,0.3270,40.85355,-73.9661,Fort Lee Near Road
2,1536,2024-01-04 05:00:00+00:00,co,0.2910,40.85355,-73.9661,Fort Lee Near Road
3,1536,2024-01-05 05:00:00+00:00,co,0.2240,40.85355,-73.9661,Fort Lee Near Road
4,1536,2024-01-06 05:00:00+00:00,co,0.2770,40.85355,-73.9661,Fort Lee Near Road
...,...,...,...,...,...,...,...
30246,1535,2024-12-28 05:00:00+00:00,no2,0.0397,40.85355,-73.9661,Fort Lee Near Road
30247,1535,2024-12-29 04:00:00+00:00,no2,0.0425,40.85355,-73.9661,Fort Lee Near Road
30248,1535,2024-12-30 05:00:00+00:00,no2,0.0233,40.85355,-73.9661,Fort Lee Near Road
30249,1535,2024-12-31 05:00:00+00:00,no2,0.0112,40.85355,-73.9661,Fort Lee Near Road


In [10]:
aq = aq[
    (aq["date"] >= "2024-01-01") &
    (aq["date"] < "2025-01-01")]
aq

,sensor_id,date,parameter,value,lat,lon,location
0,1536,2024-01-02 05:00:00+00:00,co,0.2620,40.85355,-73.9661,Fort Lee Near Road
1,1536,2024-01-03 05:00:00+00:00,co,0.3270,40.85355,-73.9661,Fort Lee Near Road
2,1536,2024-01-04 05:00:00+00:00,co,0.2910,40.85355,-73.9661,Fort Lee Near Road
3,1536,2024-01-05 05:00:00+00:00,co,0.2240,40.85355,-73.9661,Fort Lee Near Road
4,1536,2024-01-06 05:00:00+00:00,co,0.2770,40.85355,-73.9661,Fort Lee Near Road
...,...,...,...,...,...,...,...
30245,1535,2024-12-27 05:00:00+00:00,no2,0.0258,40.85355,-73.9661,Fort Lee Near Road
30246,1535,2024-12-28 05:00:00+00:00,no2,0.0397,40.85355,-73.9661,Fort Lee Near Road
30247,1535,2024-12-29 04:00:00+00:00,no2,0.0425,40.85355,-73.9661,Fort Lee Near Road
30248,1535,2024-12-30 05:00:00+00:00,no2,0.0233,40.85355,-73.9661,Fort Lee Near Road


In [11]:
aq.to_csv("openaq.csv", index=False)